# Full-duration SAM 2 vial-fill analysis

This notebook processes every vial MP4 with one reused SAM 2 predictor. It measures the real video FPS, analyzes the full sampled duration, builds a multi-frame empty-vial reference, tolerates lateral vial movement, smooths the fill signal, writes resumable per-video CSVs, and creates a combined dataset.

The existing prototype notebook is preserved. Reusable logic and tests live in `vial_analysis.py` and `test_vial_analysis.py`.

In [ ]:
from pathlib import Path
import json, os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vial_analysis import (
    AnalysisConfig, RESULT_REQUIRED_COLUMNS, analyze_batch,
    configuration_fingerprint, preview_vial_box,
)

ROOT = Path(os.getenv('VIAL_ROOT', '/home/hackathon5/vial_checks'))
VIDEO_DIR = ROOT / 'drive-download-20261031T172038Z-1-001'
WORK_DIR = ROOT / 'batch_work'
RESULTS_DIR = Path(os.getenv('VIAL_RESULTS_DIR', str(ROOT / 'batch_results')))
RUN_BATCH = os.getenv('VIAL_RUN_BATCH', '0') == '1'

# Calibrate only from a measured reference. Leave as None until known.
MM_PER_PIXEL = None
VOLUME_COEFFICIENTS = None  # e.g. (slope, intercept) for volume_ml = slope * height_mm + intercept
EMPTY_REFERENCE_MAP = ROOT / 'verified_empty_frames.json'
VERIFIED_EMPTY_FRAMES = (
    {name: tuple(indices) for name, indices in json.loads(EMPTY_REFERENCE_MAP.read_text()).items()}
    if EMPTY_REFERENCE_MAP.exists() else {}
)

config = AnalysisConfig(
    video_dir=VIDEO_DIR,
    work_dir=WORK_DIR,
    results_dir=RESULTS_DIR,
    stride=3,
    vial_box=(300, 60, 360, 420),
    reference_frame_count=15,
    minimum_reference_frames=3,
    verified_empty_frames=VERIFIED_EMPTY_FRAMES,
    require_verified_reference=False,  # False emits relative-only results when no verified map exists
    de_threshold=6.0,
    gap_rows=8,
    large_shift_px=3.0,
    smoothing_window=7,
    cycle_gap_frames=5,
    mm_per_pixel=MM_PER_PIXEL,
    volume_coefficients=VOLUME_COEFFICIENTS,
    resume=True,
)
print('Batch execution enabled:', RUN_BATCH)
print('Results:', RESULTS_DIR)

In [ ]:
videos = sorted(VIDEO_DIR.glob('*.mp4'))
inventory_rows = []
for video in videos:
    cap = cv2.VideoCapture(str(video))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    inventory_rows.append({
        'video_name': video.name, 'fps': fps, 'source_frames': frame_count,
        'duration_s': frame_count / fps if fps > 0 else np.nan,
        'sampled_frames_expected': (frame_count + config.stride - 1) // config.stride,
        'resolution': f'{width}x{height}',
    })
inventory = pd.DataFrame(inventory_rows)
display(inventory)
print(f'{len(videos)} videos | {inventory.duration_s.sum()/60:.1f} source minutes | '      f'{inventory.sampled_frames_expected.sum():,} expected sampled frames')

In [ ]:
if videos:
    fig, ax = preview_vial_box(videos[0], config.vial_box)
    plt.show()
else:
    raise FileNotFoundError(f'No MP4 files found in {VIDEO_DIR}')

## Run or resume the batch

Set `RUN_BATCH = True` in the configuration cell, or launch Jupyter with `VIAL_RUN_BATCH=1`, then run this cell. Completed videos with the same configuration fingerprint are loaded from disk instead of recomputed. A failure in one video is recorded and does not discard other results.

In [ ]:
combined_path = RESULTS_DIR / 'combined_measurements.csv'
manifest_path = RESULTS_DIR / 'processing_manifest.csv'
if RUN_BATCH:
    combined, manifest = analyze_batch(config, videos)
else:
    if combined_path.exists() or manifest_path.exists():
        summary_path = RESULTS_DIR / 'summary.json'
        if not summary_path.exists():
            raise RuntimeError('Results are incomplete: summary.json is missing.')
        summary = json.loads(summary_path.read_text())
        expected_fingerprint = configuration_fingerprint(config)
        if summary.get('config_fingerprint') != expected_fingerprint:
            raise RuntimeError('Results are stale for this configuration. Enable RUN_BATCH to regenerate them.')
        combined = pd.read_csv(combined_path)
        manifest = pd.read_csv(manifest_path)
        if not RESULT_REQUIRED_COLUMNS.issubset(combined.columns):
            raise RuntimeError('Combined results use an obsolete or incomplete schema.')
        if not combined.config_fingerprint.astype(str).eq(expected_fingerprint).all():
            raise RuntimeError('Combined results contain mixed configuration fingerprints.')
    else:
        combined, manifest = pd.DataFrame(), pd.DataFrame()
    print('Report-only mode: current results loaded; no GPU batch started.')
display(manifest)
print(f'Combined measurement rows: {len(combined):,}')

In [ ]:
if not combined.empty:
    quality = combined['status'].value_counts(dropna=False).rename_axis('status').to_frame('rows')
    quality['percent'] = 100 * quality['rows'] / len(combined)
    display(quality)
    flags = (combined['quality_flags'].fillna('').str.split(';').explode())
    flags = flags[flags.ne('')].value_counts().rename_axis('quality_flag').to_frame('rows')
    display(flags)
    display(combined['reference_basis'].value_counts().rename_axis('reference_basis').to_frame('rows'))
else:
    print('No batch results yet. Enable RUN_BATCH and execute the batch cell.')

In [ ]:
if not combined.empty:
    SELECTED_VIDEO = combined.video_name.drop_duplicates().iloc[0]  # change this to inspect another video
    group = combined[combined.video_name == SELECTED_VIDEO]
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(group.video_time_s, group.fill_height_change_smoothed_px, lw=1.5, label='change from reference')
    axes[1].plot(group.video_time_s, group.fill_height_change_monotonic_px, lw=1.5, label='cycle-aware monotonic')
    axes[0].set(title=SELECTED_VIDEO, ylabel='change height (pixels)')
    axes[1].set(ylabel='change height (pixels)', xlabel='video time (s)')
    axes[0].grid(alpha=.25); axes[1].grid(alpha=.25)
    plt.tight_layout(); plt.show()

## Physical calibration

Without `verified_empty_frames.json`, the pipeline records only `fill_height_change_px` relative to the clip's initial state. Absolute `fill_height_px`, `fill_height_mm`, and `volume_ml` remain `NaN`; the reference is labeled `relative_initial_unverified`. This prevents an already-filled vial from being reported as empty or as an absolute fill measurement.

1. Create `verified_empty_frames.json` mapping each video filename (or stem) to sampled frame indices known to show an empty vial, for example `{"video_name.mp4": [0, 1, 2]}`. Videos absent from this map remain relative-only.
2. Place an object of known height in the same image plane as the vial and measure its pixel height. Set `MM_PER_PIXEL = known_mm / measured_pixels`.
3. Collect several known liquid volumes and their measured heights. Fit a polynomial with `np.polyfit(height_mm, volume_ml, degree)` and put the returned coefficients in `VOLUME_COEFFICIENTS`.
4. Rerun the batch. Reference provenance, calibration, and pipeline version are part of the fingerprint, so stale CSVs are rejected.